# Phase 3.2: Multi-Agent Traffic Signal Control (MARL)
**Objective:** Train decentralized traffic light agents to create "green waves" for approaching emergency vehicles.

This notebook uses Independent Q-Learning with Parameter Sharing. Every signaled intersection in Kigali acts as an independent agent, observing its local queues and the presence of ambulances. All agents push their experiences into a shared memory buffer and update a single, shared neural network. This allows the entire city grid to learn cooperatively and rapidly.

In [ ]:
import sys
import random
import logging
import numpy as np
import traci
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from src.environment.manager import SimulationManager
from src.agents.traffic_marl import MultiAgentTrafficController

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

net_path = Path("../data/processed/kigali.net.xml")
route_path = Path("../data/processed/kigali_traffic.rou.xml")
marl_model_save_path = Path("../models/marl_traffic_v1.pt")

## 1. The MARL Environment Wrapper
We create a class to interface with SUMO's TraCI API specifically for traffic lights. It extracts the 3-dimensional state `[Current Phase, Max Queue Length, Ambulance Approaching Flag]` for every intersection and calculates the reward based on civilian delays and ambulance momentum.

In [ ]:
class TrafficEnvironment:
    def __init__(self, sim_manager: SimulationManager):
        self.sim = sim_manager
        self.tls_ids = []
        self.target_ambulance_id = None

    def initialize_intersections(self):
        """Finds all traffic lights in the network."""
        self.tls_ids = traci.trafficlight.getIDList()
        logging.info(f"Initialized MARL Agents on {len(self.tls_ids)} intersections.")

    def inject_ghost_ambulance(self):
        """Randomly selects a civilian car and turns it into an emergency vehicle for training."""
        vehicles = traci.vehicle.getIDList()
        if vehicles and self.target_ambulance_id not in vehicles:
            self.target_ambulance_id = random.choice(vehicles)
            traci.vehicle.setColor(self.target_ambulance_id, (255, 0, 0, 255))
            traci.vehicle.setSpeedFactor(self.target_ambulance_id, 2.0)

    def get_state(self, tls_id: str) -> np.ndarray:
        """Extracts the [Phase, Queue, Ambulance_Flag] state for a specific intersection."""
        current_phase = traci.trafficlight.getPhase(tls_id)
        
        lanes = traci.trafficlight.getControlledLanes(tls_id)
        max_queue = 0
        amb_approaching = 0.0

        for lane in lanes:
            queue = traci.lane.getLastStepHaltingNumber(lane)
            max_queue = max(max_queue, queue)
            
            vehicles_on_lane = traci.lane.getLastStepVehicleIDs(lane)
            if self.target_ambulance_id in vehicles_on_lane:
                amb_approaching = 1.0

        return np.array([current_phase / 10.0, max_queue / 50.0, amb_approaching], dtype=np.float32)

    def apply_action(self, tls_id: str, action: int):
        """Applies the agent's decision: 0 (Keep), 1 (Switch Phase)."""
        if action == 1:
            current_phase = traci.trafficlight.getPhase(tls_id)
            
            logic = traci.trafficlight.getCompleteRedYellowGreenDefinition(tls_id)[0]
            num_phases = len(logic.phases)
            
            next_phase = (current_phase + 1) % num_phases
            traci.trafficlight.setPhase(tls_id, next_phase)

    def calculate_reward(self, tls_id: str) -> float:
        """
        Rewards:
        - Small penalty for civilian queues (-0.5 per car).
        - Massive penalty if ambulance is stopped at this light (-100).
        - Massive reward if ambulance successfully clears this light (+50).
        """
        lanes = traci.trafficlight.getControlledLanes(tls_id)
        total_queue = sum([traci.lane.getLastStepHaltingNumber(lane) for lane in lanes])
        
        reward = -0.5 * total_queue

        if self.target_ambulance_id:
            for lane in lanes:
                vehicles = traci.lane.getLastStepVehicleIDs(lane)
                if self.target_ambulance_id in vehicles:
                    speed = traci.vehicle.getSpeed(self.target_ambulance_id)
                    if speed < 1.0: 
                        reward -= 100.0
                    elif speed > 5.0: 
                        reward += 50.0

        return reward

## 2. The Shared Training Loop
We step through the simulation. Every 5 simulation seconds, the traffic lights observe their environment, make a decision using the shared network, and apply it. They then store the transition in the shared memory buffer to learn.

In [ ]:
EPISODES           = 50
SIMULATION_STEPS   = 1000
DECISION_INTERVAL  = 5
BATCH_SIZE         = 128
EPSILON_START      = 1.0
EPSILON_END        = 0.05
EPSILON_DECAY      = 0.99

sim_manager = SimulationManager(net_path=net_path, route_path=route_path, use_gui=False)
env         = TrafficEnvironment(sim_manager)

agent = MultiAgentTrafficController(state_dim=3, action_dim=2, lr=1e-3)
agent.load_model(marl_model_save_path)

epsilon            = EPSILON_START
best_global_reward = -float('inf')
episode_rewards    = []   # tracked for reward-curve visualisation

print("\n--- Starting MARL Traffic Training Loop ---")
for episode in range(1, EPISODES + 1):
    try:
        sim_manager.start()
        env.initialize_intersections()

        episode_reward = 0.0

        for step in range(SIMULATION_STEPS):
            sim_manager.step()
            env.inject_ghost_ambulance()

            if step % DECISION_INTERVAL == 0:
                states  = {}
                actions = {}

                for tls_id in env.tls_ids:
                    state  = env.get_state(tls_id)
                    action = agent.select_action(state, epsilon)
                    env.apply_action(tls_id, action)
                    states[tls_id]  = state
                    actions[tls_id] = action

                sim_manager.step()

                for tls_id in env.tls_ids:
                    next_state = env.get_state(tls_id)
                    reward     = env.calculate_reward(tls_id)
                    done       = step >= SIMULATION_STEPS - DECISION_INTERVAL

                    episode_reward += reward
                    agent.push_experience(states[tls_id], actions[tls_id], reward, next_state, done)

                agent.update(BATCH_SIZE)

        if episode % 5 == 0:
            agent.update_target_network()

        epsilon = max(EPSILON_END, epsilon * EPSILON_DECAY)
        episode_rewards.append(episode_reward)

        if episode_reward > best_global_reward:
            best_global_reward = episode_reward
            agent.save_model(marl_model_save_path)

        print(f"Episode {episode:>3}/{EPISODES} | "
              f"Shared Network Reward: {episode_reward:>10.2f} | "
              f"Epsilon: {epsilon:.3f}")

    except Exception as e:
        print(f"Episode {episode} crashed: {e}")
        episode_rewards.append(float('nan'))
    finally:
        sim_manager.close()

print("--- MARL Training Complete ---")


## 3. MARL Training Stability

The reward per episode is the sum of rewards collected across all intersections.
Because the penalty for stopping an ambulance (−100) dominates early, the total
episode reward starts very negative and converges toward zero (and occasionally
positive) as agents learn to pre-emptively green-wave approaching emergency vehicles.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Filter out any NaN rewards from crashed episodes
clean_rewards = np.array([r for r in episode_rewards if not np.isnan(r)])
ep_nums       = [i + 1 for i, r in enumerate(episode_rewards) if not np.isnan(r)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('MARL Traffic Controller — Training Stability', fontsize=13, fontweight='bold')

# ── Left: episode reward ─────────────────────────────────────────────────────
axes[0].bar(ep_nums, clean_rewards,
            color=['#2ecc71' if r > 0 else '#e74c3c' for r in clean_rewards],
            edgecolor='white', linewidth=0.5)
axes[0].axhline(0, color='black', linewidth=1.0, linestyle='--')
if len(clean_rewards) >= 5:
    w = min(5, len(clean_rewards))
    ma = np.convolve(clean_rewards, np.ones(w) / w, mode='valid')
    axes[0].plot(ep_nums[w - 1:], ma, color='navy', linewidth=2.0,
                 label=f'{w}-ep moving avg')
    axes[0].legend()
axes[0].set_title('Episode Total Reward')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Cumulative Reward (all intersections)')
axes[0].grid(True, alpha=0.3)

# ── Right: epsilon decay ─────────────────────────────────────────────────────
all_epsilons = np.array([max(EPSILON_END, EPSILON_START * (EPSILON_DECAY ** i))
                         for i in range(EPISODES)])
axes[1].plot(range(1, EPISODES + 1), all_epsilons, color='purple', linewidth=2.0)
axes[1].axhline(EPSILON_END, color='gray', linestyle='--', linewidth=1.0,
                label=f'Minimum ε = {EPSILON_END}')
axes[1].set_title('Epsilon Decay (Exploration Schedule)')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Epsilon')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/04_reward_curve_marl.png', dpi=150, bbox_inches='tight')
plt.show()

positive_eps = np.sum(clean_rewards > 0)
print(f'Episodes with positive reward: {positive_eps}/{len(clean_rewards)}')
print(f'Final episode reward         : {clean_rewards[-1]:.2f}' if len(clean_rewards) else '')
